# Trees: Conceptual Overview
A __rooted tree__ $T = \left\{\mathcal{V},\mathcal{E}\right\}$ composed of nodes $\mathcal{V}$ and edges $\mathcal{E}$, is a connected, acyclic structure with a  **root** node and a parent–child hierarchy. Each node has at most one parent and zero or more children, the only node without a parent being the **root**. The edges connect nodes, making trees ideal for modeling hierarchical data (e.g., file systems, org charts, etc). 

Let's look at some key elements of a tree structure:
> **Levels and Height**: Each node in the tree is assigned a level based on its distance from the root (node with no parent). The root is at level $h=0$, its children are at $h=1$, and so on. For any node, the number of edges on the path from the root to that node defines its level. The maximum level $\max_{v\in\mathcal{V}}\left(\texttt{level}(v)\right)$ of a tree is called the height $h$, here `h = 3`.

> **Leaves and Branches**: Leaves are nodes with no children; they can occur at any level of a tree. A branch is any path from the root down through its descendants to a leaf. One such root-to-leaf path is highlighted to illustrate how height corresponds to the length of the longest branch in the tree.

Before we get into the representation of trees, let's look at some applications.

<div>
    <center>
      <img
        src="figs/Fig-General-Tree-Example.svg"
        alt="General Tree Example"
        height="400"
        width="800"
      />
    </center>
  </div>

## Applications of Trees

Trees are widely used for various applications. In everday life, trees can model various hierarchical relationships, such as: Family trees, organizational structures, file systems, and decision-making processes. In computational contexts, trees are used in data structures and algorithms, such as:

* __Binary Trees__: A tree where each node has at most two children, referred to as the left and right child. This structure is widely used, for example, in finance to represent future prices of an asset, where each node represents a possible price at a given time, and the left child represents a lower price while the right child represents a higher price. We can expand this concept to create more complex structures, e.g., where each node has $n$ children, leading to a __n-ary tree__.
* __Binary Search Tree (BST)__: A binary tree where the left child is less than the parent node, and the right child is greater than the parent node. This property allows for efficient searching, insertion, and deletion operations.
* __Heaps__: A specialized tree structure that satisfies the heap property, where the parent node is either greater than or equal to (max heap) or less than or equal to (min heap) its children. Heaps are commonly used in priority queues and sorting algorithms.
* __Tries__: A tree-like structure used for storing strings, where each node represents a character in the string. Tries are particularly useful for tasks such as autocomplete and spell checking, as they enable efficient prefix searching and retrieval of strings.

They provide a clear and intuitive way to represent relationships and dependencies among entities, making them a powerful tool for data organization and analysis.
___

<div>
    <center>
      <img
        src="figs/Fig-ThreeTree-Example-Schematic.svg"
        alt="Three Tree Example"
        height="600"
        width="800"
      />
    </center>
  </div>

## Full n-ary Trees
Ok, so trees model hierarchical relationships, but how do we store them? A common way to store trees is through __adjacency lists__. In this representation, we have the data for each node stored in a list or array, and then we have a second list or array that contains the indices of the children for each node. This allows us to efficiently traverse the tree and access the children of any node.

Let's dig into the dimensionality of a complete $n$-ary tree, and the indexing scheme of the nodes in a tree. Suppose we have an $n$-ary tree, where each node has $n$ children. Further, suppose we say the root node has index 0. Then, for any node at index $i$, the $k$-th child where $k=1,2,\ldots,n$ can be found at the index:
$$
\begin{align*}
c_{i,k} = n\;i + k
\end{align*}
$$
while the parent of the node at index $i$ can be found at:
$$\begin{align*}
p_{i} = \left\lfloor \frac{i-1}{n} \right\rfloor
\end{align*}
$$
where $\lfloor x \rfloor$ is the floor function, which rounds down to the nearest integer. This indexing scheme allows us to efficiently navigate the tree structure and access parent-child relationships. This tree will have $N$ nodes where:
$$
\boxed{
\begin{align*}
N &= \sum_{i=0}^{h} n^{i} = \frac{n^{h+1}-1}{n-1}\quad \text{for } n > 1
\end{align*}}
$$
where $h$ is the height of the tree. Alternatively, we can write the number of nodes in a complete $n$-ary tree as a function of the number of levels $l$ in the tree, where $h = L - 1$:
$$\boxed{
\begin{align*}N &= \frac{n^{L}-1}{n-1}\quad \text{for } n > 1
\end{align*}}$$
The expressions gives us the total number of nodes in a complete $n$-ary tree, which can be useful for understanding the stotrage requiements of the tree structure. 

## Recombining Tree
Many of the trees that we will be interested are recombining trees, e.g., paths that visit the same multiset of moves collapse into one node. In this case, the growth in the number of nodes goes from exponential in the full $n$-ary tree case down to polynomial in the number of steps.

Finish me. 

In [78]:
n = 3; # ternary tree
price = 100.0;
h = 2; # height of the tree
Δt = (1/252); # time step
u = exp(0.1*Δt); # up factor
d = exp(-0.1*Δt); # down factor

In [70]:
d*u

1.0

In [83]:
# ── combinatorics helpers ──────────────────────────────────────────────────────
nodes_at_level(i::Integer, n::Integer) = binomial(i + n - 1, i)

level_offset(i::Integer, n::Integer) = i == 0 ? 0 : binomial(i + n - 1, i - 1) # start of level i

# Find level i s.t. offset(i) ≤ j < offset(i+1)
function level_of(j::Integer, n::Integer)
    @assert j ≥ 0 && n ≥ 2
    i = 0
    while true
        next_off = binomial(i + n, i)   # = level_offset(i+1, n)
        if j < next_off
            return i
        end
        i += 1
    end
end

# Unrank: given rank r at level i, return composition k⃗ (length n, sum = i), lex order
function unrank_comp(r::Integer, i::Integer, n::Integer)
    k = zeros(Int, n)
    rem = i
    rcur = r
    for m in 1:(n-1)
        km = 0
        while true
            c = binomial(rem - km + (n - m) - 1, (n - m) - 1)
            if rcur < c
                break
            end
            rcur -= c
            km += 1
        end
        k[m] = km
        rem -= km
    end
    k[n] = rem
    return k
end

# Rank: given composition k⃗ at level i, return its 0-based rank within that level (lex order)
function rank_comp(k::AbstractVector{<:Integer}, i::Integer, n::Integer)
    @assert length(k) == n
    @assert sum(k) == i
    rank = 0
    rem = i
    for m in 1:(n-1)
        km = k[m]
        for j in 0:(km-1)
            rank += binomial(rem - j + (n - m) - 1, (n - m) - 1)
        end
        rem -= km
    end
    return rank
end

# ── main API ───────────────────────────────────────────────────────────────────
"""
    children_indices(j, n; base=0)

Return the flat-array indices of the n children of node `j` in a recombining n-ary tree.
`base=0` returns 0-based indices (add 1 if you want Julia 1-based positions).
"""
function children_indices(j::Integer, n::Integer; base::Integer=0)
    i  = level_of(j, n)
    r  = j - level_offset(i, n)              # rank within level i
    k  = unrank_comp(r, i, n)                # counts (k1,…,kn), sum = i
    off_next = level_offset(i + 1, n)

    out = Vector{Int}(undef, n)
    for m in 1:n
        k[m] += 1                            # bump one move for the child
        rc = rank_comp(k, i + 1, n)          # rank at next level
        out[m] = off_next + rc + base
        k[m] -= 1                            # restore
    end
    return out |> sort
end

function index_counts(j::Integer, n::Integer)
    i = level_of(j, n)
    r = j - level_offset(i, n)
    return i, unrank_comp(r, i, n) |> reverse
end

index_counts (generic function with 1 method)

Fill me in

In [93]:
children_indices(3,3)

3-element Vector{Int64}:
 7
 8
 9

In [103]:
binomial(2 + n, 2) # number of nodes in the tree --- IGNORE ---

10

In [ ]:
P, connectivity = let
    # initialize -
    P = Dict{Int64,Float64}() 
    connectivity = Dict{Int64, Array{Int64,1}}()
    Nₕ = binomial(h + n, h) # number of nodes in the tree

    P[0] = price # set root price
    Δ = [u, 1, d]  # up, stay, down factors
    number_of_moves = length(Δ);
    for i ∈ 0:(Nₕ - 1)
        connectivity[i] = children_indices(i, n; base=0)
        (_, k) = index_counts(i, n)

        tmp = 1.0;
        for i ∈ 1:number_of_moves
            tmp *= Δ[i]^(k[i]) # accumulate the price factor for each move
        end
        P[i] = price * tmp; # price at node i
    end

    P, connectivity    
end;

In [102]:
connectivity

Dict{Int64, Vector{Int64}} with 10 entries:
  0 => [1, 2, 3]
  4 => [10, 11, 14]
  5 => [11, 12, 15]
  6 => [12, 13, 16]
  2 => [5, 6, 8]
  7 => [14, 15, 17]
  9 => [17, 18, 19]
  8 => [15, 16, 18]
  3 => [7, 8, 9]
  1 => [4, 5, 7]

In [105]:
P

Dict{Int64, Float64} with 10 entries:
  0 => 100.0
  4 => 100.079
  5 => 100.04
  6 => 100.0
  2 => 100.0
  7 => 100.0
  9 => 99.9207
  8 => 99.9603
  3 => 99.9603
  1 => 100.04

In [98]:
index_counts(8, n)

(2, [0, 1, 1])

## Traversal Algorithms
We visit (traverse) the nodes in a tree differently than in an array. Let's look at two common traversal algorithms for trees (that we'll use as building blocks for other algorithms): depth-first search (DFS) and breadth-first search (BFS).

### Depth-First Search (DFS)
Depth-first search (DFS) __recursively__ explores as far as possible along each branch before backtracking. It uses a `Set` data structure to keep track of the nodes that have already been visited. 

The algorithm starts at a given node, marks it as visited, and then recursively visits each unvisited neighbor until all reachable nodes are visited. If a node has no unvisited neighbors, the algorithm backtracks to the previous node and continues exploring from there. Let's look at a simple recursive implementation of a DFS algorithm.

__Initialization__: Given a graph $\mathcal{G} = (\mathcal{V}, \mathcal{E})$, a starting vertex $v_{s}\in\mathcal{V}$, and an empty set of visited vertices $\mathcal{V}_{\text{visited}}$.

1. If $v_{s}\notin\mathcal{V}_{\text{visited}}$, then:
    - Add $v_{s}$ to the $\mathcal{V}_{\text{visited}}$ set: $\mathcal{V}_{\text{visited}}\gets\mathcal{V}_{\text{visited}}\cup\{v_{s}\}$.
    - Get the neighbors of node $v_{s}$: Set $\mathcal{N}_{s} \gets \texttt{neighbors}(v_{s})$.
    - For each neighbor $v_{n}\in\mathcal{N}_{s}$, do:
        - __Recursively__ call the DFS algorithm with $v_{n}$ as the new starting vertex. (Goto step 1 with $v_{n}$ as the new starting vertex.)
2. If $v_{s}\in\mathcal{V}_{\text{visited}}$, then return.


DFS runs in $\mathcal{O}(|\mathcal{V}|+|\mathcal{E}|)$ time, where $|\mathcal{V}|$ is the number of vertices and $|\mathcal{E}|$ is the number of edges in the graph. It uses $\mathcal{O}(|\mathcal{V}|)$ space for the visited set (plus recursion depth).


### Breadth-First Search (BFS)
Breadth-first search (BFS) is a traversal algorithm that explores all the neighbors of a node before moving on to the next level of nodes. It uses a `Queue` data structure to keep track of the nodes to visit next. 

The algorithm starts at a given node, marks it as visited, and then enqueues all its unvisited neighbors. It continues to dequeue nodes from the front of the queue, marking them as visited and enqueuing their unvisited neighbors, until all reachable nodes are visited. Let's look at an implementation of a BFS algorithm:

__Initialization__: Given a graph $\mathcal{G} = (\mathcal{V}, \mathcal{E})$, a starting vertex $v_{s}\in\mathcal{V}$, and an empty Set of visited vertices $\mathcal{V}_{\text{visited}}$, and an empty queue $\mathcal{Q}$.

1. Add the starting vertex $v_{s}$ to the queue: $\mathcal{Q}\gets\texttt{enqueue}(\mathcal{Q}, v_{s})$.
2. While the queue $\mathcal{Q}$ is not empty, __do__:
    - Dequeue a vertex $v_{n}$ from the front of the queue: $v_{n}\gets\texttt{dequeue}(\mathcal{Q})$.
    - If $v_{n}\notin\mathcal{V}_{\text{visited}}$, __then__:
        - Add $v_{n}$ to the $\mathcal{V}_{\text{visited}}$ set: $\mathcal{V}_{\text{visited}}\gets\mathcal{V}_{\text{visited}}\cup\{v_{n}\}$.
        - Get the neighbors of node $v_{n}$: Set $\mathcal{N}_{n} \gets \texttt{neighbors}(v_{n})$.
        - For each neighbor $v_{m}\in\mathcal{N}_{n}$, do:
            - If $v_{m}\notin\mathcal{V}_{\text{visited}}$, then enqueue it: $\mathcal{Q}\gets\texttt{enqueue}(\mathcal{Q}, v_{m})$.

BFS runs in $\mathcal{O}(|\mathcal{V}|+|\mathcal{E}|)$ time, where $|\mathcal{V}|$ is the number of vertices and $|\mathcal{E}|$ is the number of edges in the graph. It uses $\mathcal{O}(|\mathcal{V}|)$ space for the visited set (plus queue space).

____